### ACDP Weighting Cause of No SMOTE

In [19]:
import pandas as pd
import numpy as np
from graphviz import Digraph
import os

# 1. Targeting Path
file_path = 'data/diabetes_preprocessed_privacy_copy.csv'

# 2. Reading Database
df_privacy = pd.read_csv(file_path)

# 3. Verifikasi Data
print("System: Data berhasil di-load.")
df_privacy.head(100)

System: Data berhasil di-load.


,Diabetes_012,HighBP,HighChol,CholCheck,BMI,Smoker,Stroke,HeartDiseaseorAttack,PhysActivity,Fruits,...,DiffWalk,Sex,Age,Education,Income,Age_Group,BMI_Group,Income_Group,Education_Group,GenHlth_Group
0,0,1,1,1,40.0,1,0,0,0,0,...,1,0,9,4,3,MiddleAge,Obese,Low,MidEdu,Poor
1,0,0,0,0,25.0,1,0,0,1,0,...,0,0,7,6,1,Adult,Overweight,Low,HighEdu,Fair
2,0,1,1,1,28.0,0,0,0,0,1,...,1,0,9,4,8,MiddleAge,Overweight,High,MidEdu,Poor
3,0,1,0,1,27.0,0,0,0,1,1,...,0,0,11,3,6,Senior,Overweight,Middle,MidEdu,Good
4,0,1,1,1,24.0,0,0,0,1,1,...,0,0,11,5,4,Senior,Normal,Middle,HighEdu,Good
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,2,1,1,1,25.0,1,0,1,0,1,...,1,0,9,2,3,MiddleAge,Overweight,Low,LowEdu,Poor
96,2,0,0,1,32.0,0,0,0,1,0,...,0,0,3,5,3,Young,Obese,Low,HighEdu,Poor
97,0,1,0,1,44.0,0,0,0,1,1,...,0,0,9,4,6,MiddleAge,Obese,Middle,MidEdu,Fair
98,0,1,1,1,28.0,0,0,0,0,1,...,1,0,11,4,3,Senior,Overweight,Low,MidEdu,Good


### Logic

In [20]:


# Sesuaikan PATH Graphviz milikmu jika diperlukan
os.environ["PATH"] += os.pathsep + 'C:/Program Files/Graphviz/bin'

# ==========================================
# 1. DUMMY SCOUTER (Slot untuk temanmu)
# ==========================================
def dummy_scouter(feature_data, target_data):
    """
    Ini adalah fungsi placeholder. 
    Nanti temanmu harus me-replace ini dengan algoritma ACE miliknya.
    """
    return np.random.rand() # Memberikan nilai korelasi acak agar tree bisa di-render

# ==========================================
# 2. CORE ACDP-TREE (Raw Sorter Mode)
# ==========================================
def get_raw_counts(target_data):
    """Fase Looting: Mengumpulkan jumlah riil pasien di leaf node ini."""
    real_counts = target_data.value_counts().to_dict()
    for cls in [0.0, 1.0, 2.0]:
        if cls not in real_counts:
            real_counts[cls] = 0
    return real_counts

def build_acdp_tree_base(data, target_col, features, depth=0, max_depth=3):
    # Base Case
    if len(data) == 0: return None
    if depth >= max_depth or len(data[target_col].unique()) == 1:
        return get_raw_counts(data[target_col])

    # Fase Scouting (Mencari atribut pembelah terbaik)
    best_ac = -1
    best_feat = None
    for feat in features:
        # Panggil Dummy Scouter (Nanti ganti dengan modul ACE)
        ac = dummy_scouter(data[feat], data[target_col])
        if ac > best_ac:
            best_ac = ac
            best_feat = feat
            
    if best_feat is None:
        return get_raw_counts(data[target_col])

    # Fase Breaching (Membelah data ke layer berikutnya)
    node = {'feature': best_feat, 'children': {}, 'depth': depth}
    remaining_feats = [f for f in features if f != best_feat]
    
    for val in data[best_feat].unique():
        subset = data[data[best_feat] == val]
        node['children'][val] = build_acdp_tree_base(
            subset, target_col, remaining_feats, depth + 1, max_depth
        )
            
    return node

### Visual

In [21]:
# ==========================================
# 3. VISUALIZATION (Map Rendering)
# ==========================================
def visualize_tree_raw(tree, feature_names):
    dot = Digraph(comment='ACDP Base Tree')
    dot.attr(rankdir='LR', size='15,20', dpi='300')
    
    def add_nodes(node, parent_name=None, edge_label=None):
        # 1. Security Check: Jika node kosong (Data Void)
        if node is None:
            return

        # 2. Logical Check: Jika tidak ada kunci 'feature', berarti ini adalah Leaf
        if 'feature' not in node: 
            # Ini adalah Leaf. Menampilkan raw count.
            # Menggunakan .get(key, 0) agar aman jika ada kelas yang hilang
            c0 = node.get(0.0, 0)
            c1 = node.get(1.0, 0)
            c2 = node.get(2.0, 0)
            
            label = f"RAW COUNT (Wait for Laplace):\nClass 0: {c0}\nClass 1: {c1}\nClass 2: {c2}"
            color = 'lightgray' 
            
            node_name = str(np.random.rand())
            dot.node(node_name, label, shape='box', style='filled', fillcolor=color)
            if parent_name:
                dot.edge(parent_name, node_name, label=edge_label)
            return

        # 3. Decision Node: Jika ada kunci 'feature'
        node_name = str(np.random.rand())
        dot.node(node_name, f"Split by:\n{node['feature']}", shape='ellipse')
        
        if parent_name:
            dot.edge(parent_name, node_name, label=edge_label)
            
        # Rekursi ke anak-anak pohon (Children)
        for val, child in node['children'].items():
            add_nodes(child, node_name, edge_label=str(val))

    add_nodes(tree)
    return dot

# ==========================================
# 4. EXECUTION PATH
# ==========================================
# Pastikan df_privacy sudah ter-load sebelumnya
print("System: Compiling Tree Architecture...")

features_list = ['Age_Group', 'BMI_Group', 'Income_Group', 'Education_Group', 'GenHlth_Group']

# Eksekusi fungsi pembentuk pohon
my_acdp_base = build_acdp_tree_base(df_privacy, 'Diabetes_012', features_list, max_depth=3)

# Eksekusi visualisasi
graph = visualize_tree_raw(my_acdp_base, features_list)
graph.render('acdp_tree_raw_architecture', format='pdf', view=True)

print("System: Rendering Complete. PDF generated.")

System: Compiling Tree Architecture...
System: Rendering Complete. PDF generated.
